# Top-k / Top-p Sampling

**难度:** Medium

实现语言模型解码的 top-k / top-p（核）采样。

这些采样策略在采样前将词表过滤为高概率 token，在文本生成中平衡多样性和质量。

**签名:** `sample_top_k_top_p(logits, top_k=0, top_p=1.0, temperature=1.0) -> int`

**参数:**
- `logits` — 词表上的原始 logits (V,)
- `top_k` — 仅保留 top-k 个 token（0 = 禁用）
- `top_p` — 保留累积概率 <= p 的 token（1.0 = 禁用）
- `temperature` — 温度缩放

**返回:** 采样的 token 索引（整数）

**约束:**
- 先应用温度，再 top-k，再 top-p
- `top_k=1` 必须始终返回 argmax

**提示:**
1. logits /= temperature
2. top-k：将低于第 k 大值的 logits 设为 -inf
3. top-p：降序排列 → softmax 概率 cumsum → 遮蔽 cumsum > p 的部分 → 设为 -inf
4. 从 softmax(logits) 中采样

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [ ]:
# ✅ SOLUTION

def sample_top_k_top_p(logits, top_k=0, top_p=1.0, temperature=1.0):
    # ---- Step 1: 温度缩放 ----
    # 温度 T 控制分布的"尖锐程度"
    # T < 1: logits 被放大 → softmax 更尖锐 → 更确定性（趋向 argmax）
    # T > 1: logits 被缩小 → softmax 更平坦 → 更随机
    # T → 0: 等价于 argmax（贪心解码）
    logits = logits / max(temperature, 1e-8)

    # ---- Step 2: Top-k 过滤 ----
    # 只保留概率最高的 k 个 token，其余设为 -inf（softmax 后概率为 0）
    if top_k > 0:
        # topk(k) 返回最大的 k 个值，取其中最小的作为阈值
        top_k_val = logits.topk(top_k).values[-1]
        logits[logits < top_k_val] = float('-inf')

    # ---- Step 3: Top-p（核采样）过滤 ----
    # 按概率从高到低排序，累加概率直到超过 p，截断剩余
    if top_p < 1.0:
        # 降序排列 logits
        sorted_logits, sorted_idx = torch.sort(logits, descending=True)
        # 计算 softmax 概率
        probs = torch.softmax(sorted_logits, dim=-1)
        # 累积概率
        cumsum = torch.cumsum(probs, dim=-1)
        # mask: 累积概率（减去自身）> p 的位置需要被遮蔽
        # 为什么要减去 probs？因为我们要保留"加上这个 token 后刚好不超过 p"的那个
        mask = (cumsum - probs) > top_p
        sorted_logits[mask] = float('-inf')
        # scatter_ 把排序后的结果映射回原始索引位置
        logits = torch.empty_like(logits).scatter_(0, sorted_idx, sorted_logits)

    # ---- Step 4: 采样 ----
    # 从过滤后的分布中采样一个 token
    probs = torch.softmax(logits, dim=-1)
    return torch.multinomial(probs, 1).item()

In [ ]:
# Demo
logits = torch.tensor([1.0, 5.0, 2.0, 0.5])
print('top_k=1:', sample_top_k_top_p(logits.clone(), top_k=1))
print('top_p=0.5:', sample_top_k_top_p(logits.clone(), top_p=0.5))

In [ ]:
from torch_judge import check
check('topk_sampling')

## 📝 核心思路总结

1. **Temperature 控制随机性**：T<1 更确定（尖锐），T>1 更随机（平坦），T→0 等价于 argmax
2. **Top-k 截断低概率尾部**：只保留概率最高的 k 个 token，简单但固定候选数
3. **Top-p（核采样）自适应候选数**：按概率从高到低累加，直到超过 p 为止，候选数动态变化